# Societe Generale Rates

Momentum strategies for rates: Bridging the Gap between statistics and option theory

In [1]:
import os
import numpy as np
import pandas as pd
from   scipy.stats import norm

In [2]:
write_path = os.getcwd()
res_path   = os.path.abspath(os.path.join(write_path, ".."))
repo_path  = os.path.abspath(os.path.join(res_path, ".."))
data_path  = os.path.join(repo_path, "data")
fut_path   = os.path.join(data_path, "FuturesData")

# Momentum strategies for rates: Bridging the Gap between statistics and option theory

First start by getting the universe of fixed income and STIR products. The author use all the rates and STIR futures available at the time. 

In [3]:
guide_path      = os.path.join(data_path, "TickerGuide.xlsx")
df_ticker_guide = (pd
    .read_excel(io = guide_path, sheet_name = "fut_guide")
    .loc[lambda x: x.group.isin(["fixed_income", "stir"])]
    [["Tmp", "Root Contract", "Front", "group"]]
    .rename(
        columns = {
            "Tmp"          : "name",
            "Root Contract": "root",
            "Front"        : "front"})
    .assign(ticker = lambda x: x.front.str.replace(" 1", "1").str.lower().str.replace(" ", "_")))

ticker_dict = (df_ticker_guide
    .set_index("ticker")
    .group
    .to_dict())

tickers = list(ticker_dict.keys())

In [4]:
px_path = os.path.join(fut_path, "PrepFuturesPX.parquet")
df_px   = (pd
    .read_parquet(path = px_path, engine = "pyarrow")
    .loc[lambda x: x.ticker.isin(tickers)]
    [["date", "ticker", "adj_val"]]
    .assign(group = lambda x: x.ticker.map(ticker_dict))
    .dropna()
    .rename(columns = {"adj_val": "px"}))

The author defines two types of signals for trend following, <br> 
one called past return indicator
\begin{equation}
\mathbf{Trend} = \frac{1}{T-t} \left(\ln(S_T) - \ln(S_t) \right)
\end{equation}
The other is called regression line
\begin{equation}
\mathbf{Trend} = \frac{\textrm{cov}(ln(S_s), s)}{\textrm{Var}(s)} / s
\end{equation}

The author uses a trend window of 100 days, and then uses a z-score window, but doesn't state what the window of the z-score is. In this case we'll start with full-sample in-sample

In [5]:
def _get_past_rtn(df: pd.DataFrame, window: int) -> pd.DataFrame: 

    df_out =(df
        .set_index("date")
        .sort_index()
        .assign(
            log_px  = lambda x: np.log(x.px),
            signal  = lambda x: 1 / (window) * (x.log_px.diff(window)),
            z_score = lambda x: (x.signal - x.signal.mean()) / x.signal.std(),
            prob    = lambda x: norm.cdf(x.z_score))
        .assign(
            lag_zscore = lambda x: x.z_score.shift(),
            lag_prob   = lambda x: x.prob.shift())
        .dropna())

    return df_out

window = 100

df_past_signal = (df_px
    .groupby("ticker")
    .apply(_get_past_rtn, window)
    .reset_index())

In [6]:
def _get_regression_signal(df: pd.DataFrame, window: int = 100) -> pd.DataFrame: 

    df_out = (df
        .sort_values("date")
        .reset_index(drop = True)
        .reset_index()
        .rename(columns = {"index": "x"})
        .assign(
            log_px = lambda x: np.log(x.px),
            x      = lambda x: x.x + 1,
            signal = lambda x: x.log_px.rolling(window = window).cov(x.x) / x.log_px.rolling(window = window).var())
        .set_index("date"))

    return df_out

df_regress_signal = (df_px
    .groupby("ticker")
    .apply(_get_regression_signal, window)
    .reset_index()
    .drop(columns = ["x"]))